# Klasifikasi Kematangan Tapai - Support Vector Machine (SVM)

Notebook ini menggunakan **Support Vector Machine (SVM)** dari scikit-learn untuk memprediksi status kematangan tapai berdasarkan data sensor.

## Dataset
- **Fitur:** `jam`, `suhu`, `kelembaban`, `kadar_gas`
- **Target:** `status_kematangan` (belum matang, matang, terlalu matang)
- **10 percobaan**, masing-masing 60 jam pengamatan

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.decomposition import PCA

plt.style.use('seaborn-v0_8')
print('Libraries loaded successfully')

## 1. Load & Eksplorasi Data

In [ ]:
df = pd.read_csv('dataset/dataset_kematangan_tapai_v2.csv')
print('Shape:', df.shape)
df.head(10)

In [ ]:
print('Distribusi label:')
counts = df['status_kematangan'].value_counts()
print(counts)

fig, ax = plt.subplots(figsize=(7, 4))
counts.plot(kind='bar', color=['#3498db', '#2ecc71', '#e74c3c'], edgecolor='white', ax=ax)
ax.set_title('Distribusi Status Kematangan')
ax.set_xlabel('Status')
ax.set_ylabel('Jumlah Sampel')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('plot_svm_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

## 2. Preprocessing

In [ ]:
label_map = {'belum matang': 0, 'matang': 1, 'terlalu matang': 2}
df['label'] = df['status_kematangan'].map(label_map)

X = df[['jam', 'suhu', 'kelembaban', 'kadar_gas']].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}')

## 3. Training SVM dengan Berbagai Kernel

In [ ]:
kernels = ['linear', 'rbf', 'poly', 'sigmoid']
results = {}

for kernel in kernels:
    svm = SVC(kernel=kernel, C=1.0, random_state=42)
    svm.fit(X_train_scaled, y_train)
    acc = accuracy_score(y_test, svm.predict(X_test_scaled))
    cv_scores = cross_val_score(svm, X_train_scaled, y_train, cv=5)
    results[kernel] = {'accuracy': acc, 'cv_mean': cv_scores.mean(), 'cv_std': cv_scores.std()}
    print(f'Kernel={kernel:8s} | Test Acc: {acc:.4f} | CV: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

best_kernel = max(results, key=lambda k: results[k]['accuracy'])
print(f'\nBest kernel: {best_kernel} (accuracy={results[best_kernel]["accuracy"]:.4f})')

## 4. Hyperparameter Tuning (GridSearch)

In [ ]:
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.01, 0.1]
}

grid_search = GridSearchCV(
    SVC(kernel='rbf', random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train_scaled, y_train)

print(f'Best params: {grid_search.best_params_}')
print(f'Best CV score: {grid_search.best_score_:.4f}')

In [ ]:
best_svm = grid_search.best_estimator_
y_pred = best_svm.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)

print(f'Test Accuracy (best SVM): {accuracy:.4f} ({accuracy*100:.2f}%)')

target_names = ['belum matang', 'matang', 'terlalu matang']
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=target_names))

## 5. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=target_names, yticklabels=target_names, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix - SVM (RBF Kernel)\nAccuracy: {accuracy*100:.2f}%')
plt.tight_layout()
plt.savefig('plot_svm_cm.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Visualisasi Decision Boundary (PCA 2D)

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(scaler.transform(X))

svm_2d = SVC(kernel='rbf', C=grid_search.best_params_['C'],
             gamma=grid_search.best_params_['gamma'], random_state=42)
svm_2d.fit(X_pca, y)

h = 0.05
x_min, x_max = X_pca[:, 0].min() - 1, X_pca[:, 0].max() + 1
y_min, y_max = X_pca[:, 1].min() - 1, X_pca[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = svm_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(9, 6))
ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
scatter_colors = ['#3498db', '#2ecc71', '#e74c3c']
for i, (label, color) in enumerate(zip(target_names, scatter_colors)):
    mask = y == i
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], c=color, label=label, s=20, alpha=0.7)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('SVM Decision Boundary (PCA 2D)')
ax.legend()
plt.tight_layout()
plt.savefig('plot_svm_boundary.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Kesimpulan

SVM dengan kernel RBF mampu menangkap **batas keputusan non-linear** antara kelas kematangan. Namun, seperti Logistic Regression, SVM juga memperlakukan setiap observasi jam secara **independen** tanpa mempertimbangkan konteks urutan waktu fermentasi sebelumnya — yang merupakan keterbatasan signifikan untuk data deret waktu ini.